In [1]:
import os
import re
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from dataclasses import dataclass, field
from typing import List

from google.colab import files

print("Environment ready.")

Environment ready.


In [2]:
uploaded = files.upload()

print("\nUploaded files:")
for filename in uploaded.keys():
    print(" -", filename)

Saving category_tree.csv to category_tree.csv
Saving events.csv to events.csv
Saving generate_sample.py to generate_sample.py
Saving item_properties_part1.csv to item_properties_part1.csv
Saving item_properties_part2.csv to item_properties_part2.csv

Uploaded files:
 - category_tree.csv
 - events.csv
 - generate_sample.py
 - item_properties_part1.csv
 - item_properties_part2.csv


In [3]:
all_files = os.listdir("/content")

print("Files detected:\n")

for f in all_files:
    if f.endswith(".csv"):
        print(f)

Files detected:

item_properties_part2.csv
events.csv
category_tree.csv
item_properties_part1.csv


In [4]:
def find_file(possible_names):

    for name in possible_names:

        path = "/content/" + name

        if os.path.exists(path):
            return path

    return None


events_path = find_file([
    "events.csv"
])

properties1_path = find_file([
    "item_properties_part1.csv",
    "item_properties1.csv",
    "item_properties_1.csv"
])

properties2_path = find_file([
    "item_properties_part2.csv",
    "item_properties2.csv",
    "item_properties_2.csv"
])

category_path = find_file([
    "category_tree.csv"
])

sample_path = find_file([
    "generate_sample.csv",
    "generate_sample"
])


print("Events:", events_path)
print("Properties 1:", properties1_path)
print("Properties 2:", properties2_path)
print("Category tree:", category_path)
print("Generate sample:", sample_path)

Events: /content/events.csv
Properties 1: /content/item_properties_part1.csv
Properties 2: /content/item_properties_part2.csv
Category tree: /content/category_tree.csv
Generate sample: None


In [5]:
events = pd.read_csv(events_path)

print("EVENTS")
print("=" * 50)

print("Shape:", events.shape)
print("Columns:", events.columns.tolist())

display(events.head())

EVENTS
Shape: (15000, 5)
Columns: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']


,timestamp,visitorid,event,itemid,transactionid
0,1432224088240,1373,view,1125,NaN
1,1431446475849,725,view,1433,NaN
2,1431210817561,1984,transaction,1294,53132.0
3,1432006761568,640,view,1360,NaN
4,1431403829105,1184,view,1519,NaN


In [6]:
properties1 = pd.read_csv(
    properties1_path
)

properties2 = pd.read_csv(
    properties2_path
)

print("ITEM PROPERTIES PART 1")
print(properties1.shape)
print(properties1.columns.tolist())

print("\nITEM PROPERTIES PART 2")
print(properties2.shape)
print(properties2.columns.tolist())

ITEM PROPERTIES PART 1
(6000, 4)
['timestamp', 'itemid', 'property', 'value']

ITEM PROPERTIES PART 2
(6000, 4)
['timestamp', 'itemid', 'property', 'value']


In [7]:
item_properties = pd.concat(
    [
        properties1,
        properties2
    ],
    ignore_index=True
)

print(
    "Combined item properties:",
    item_properties.shape
)

display(
    item_properties.head()
)

Combined item properties: (12000, 4)


,timestamp,itemid,property,value
0,1430622000000,1000,categoryid,18
1,1430622000000,1000,available,1
2,1430622000000,1000,790,n1774.245
3,1430622000000,1000,6,n292.003
4,1430622000000,1000,364,n650.090


In [8]:
category_tree = pd.read_csv(
    category_path
)

print("CATEGORY TREE")
print("=" * 50)

print("Shape:", category_tree.shape)
print("Columns:", category_tree.columns.tolist())

display(category_tree.head())

CATEGORY TREE
Shape: (40, 2)
Columns: ['categoryid', 'parentid']


,categoryid,parentid
0,1,NaN
1,2,NaN
2,3,NaN
3,4,NaN
4,5,NaN


In [9]:
if sample_path is not None:

    try:

        generate_sample = pd.read_csv(
            sample_path
        )

        print(
            "GENERATE SAMPLE"
        )

        print(
            "Shape:",
            generate_sample.shape
        )

        print(
            "Columns:",
            generate_sample.columns.tolist()
        )

        display(
            generate_sample.head()
        )

    except Exception as e:

        generate_sample = None

        print(
            "Could not load generate_sample:",
            e
        )

else:

    generate_sample = None

    print(
        "generate_sample not found."
    )

generate_sample not found.


In [10]:
datasets = {
    "events": events,
    "item_properties": item_properties,
    "category_tree": category_tree
}

if generate_sample is not None:
    datasets["generate_sample"] = generate_sample


for name, df in datasets.items():

    print("\n")
    print("=" * 70)
    print(name.upper())
    print("=" * 70)

    print("Shape:", df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nMissing values:")
    print(
        df.isna()
        .sum()
        .sort_values(ascending=False)
        .head(10)
    )



EVENTS
Shape: (15000, 5)

Columns:
['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']

Missing values:
transactionid    13483
timestamp            0
visitorid            0
event                0
itemid               0
dtype: int64


ITEM_PROPERTIES
Shape: (12000, 4)

Columns:
['timestamp', 'itemid', 'property', 'value']

Missing values:
timestamp    0
itemid       0
property     0
value        0
dtype: int64


CATEGORY_TREE
Shape: (40, 2)

Columns:
['categoryid', 'parentid']

Missing values:
parentid      5
categoryid    0
dtype: int64


In [11]:
events = events.copy()

events["itemid"] = (
    events["itemid"]
    .astype(str)
)

events["visitorid"] = (
    events["visitorid"]
    .astype(str)
)

if "timestamp" in events.columns:

    events["timestamp"] = pd.to_datetime(
        events["timestamp"],
        unit="ms",
        errors="coerce"
    )

events["event"] = (
    events["event"]
    .astype(str)
    .str.lower()
)

print(
    events["event"]
    .value_counts()
)

event
view           10557
addtocart       2926
transaction     1517
Name: count, dtype: int64


In [12]:
def is_event_dataset(df):

    if df is None:
        return False

    required = {
        "visitorid",
        "itemid",
        "event"
    }

    return required.issubset(
        set(df.columns)
    )


if is_event_dataset(generate_sample):

    sample_events = generate_sample.copy()

    sample_events["itemid"] = (
        sample_events["itemid"]
        .astype(str)
    )

    sample_events["visitorid"] = (
        sample_events["visitorid"]
        .astype(str)
    )

    sample_events["event"] = (
        sample_events["event"]
        .astype(str)
        .str.lower()
    )

    interaction_events = pd.concat(
        [
            events,
            sample_events
        ],
        ignore_index=True
    )

    print(
        "generate_sample contains event data."
    )

else:

    interaction_events = events.copy()

    print(
        "generate_sample is not an event table."
    )

print(
    "Final interaction dataset:",
    interaction_events.shape
)

generate_sample is not an event table.
Final interaction dataset: (15000, 5)


In [13]:
interaction_events = (
    interaction_events
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    "Interactions after deduplication:",
    interaction_events.shape
)

Interactions after deduplication: (15000, 5)


In [14]:
product_behavior = (
    interaction_events
    .groupby("itemid")
    .agg(

        views=(
            "event",
            lambda x:
            (x == "view").sum()
        ),

        add_to_cart=(
            "event",
            lambda x:
            (x == "addtocart").sum()
        ),

        transactions=(
            "event",
            lambda x:
            (x == "transaction").sum()
        )
    )
    .reset_index()
)

display(
    product_behavior.head()
)

,itemid,views,add_to_cart,transactions
0,1000,19,4,1
1,1001,12,5,3
2,1002,24,2,1
3,1003,18,7,5
4,1004,19,5,1


In [15]:
product_behavior["conversion_rate"] = (

    product_behavior["transactions"]

    /

    product_behavior["views"]
    .replace(0, np.nan)

).fillna(0)


product_behavior["cart_rate"] = (

    product_behavior["add_to_cart"]

    /

    product_behavior["views"]
    .replace(0, np.nan)

).fillna(0)


display(
    product_behavior.head()
)

,itemid,views,add_to_cart,transactions,conversion_rate,cart_rate
0,1000,19,4,1,0.052632,0.210526
1,1001,12,5,3,0.250000,0.416667
2,1002,24,2,1,0.041667,0.083333
3,1003,18,7,5,0.277778,0.388889
4,1004,19,5,1,0.052632,0.263158


In [16]:
ALPHA = 5
BETA = 20

product_behavior["smoothed_conversion"] = (

    product_behavior["transactions"]
    + ALPHA

) / (

    product_behavior["views"]
    + ALPHA
    + BETA
)

display(
    product_behavior[
        [
            "itemid",
            "views",
            "transactions",
            "conversion_rate",
            "smoothed_conversion"
        ]
    ].head(20)
)

,itemid,views,transactions,conversion_rate,smoothed_conversion
0,1000,19,1,0.052632,0.136364
1,1001,12,3,0.250000,0.216216
2,1002,24,1,0.041667,0.122449
3,1003,18,5,0.277778,0.232558
4,1004,19,1,0.052632,0.136364
5,1005,14,5,0.357143,0.256410
6,1006,13,2,0.153846,0.184211
7,1007,19,2,0.105263,0.159091
8,1008,24,4,0.166667,0.183673
9,1009,20,3,0.150000,0.177778


In [17]:
max_views = (
    product_behavior["views"].max()
)

if max_views > 0:

    product_behavior["popularity"] = (

        product_behavior["views"]
        / max_views

    )

else:

    product_behavior["popularity"] = 0


display(
    product_behavior[
        [
            "itemid",
            "views",
            "popularity"
        ]
    ].head()
)

,itemid,views,popularity
0,1000,19,0.633333
1,1001,12,0.400000
2,1002,24,0.800000
3,1003,18,0.600000
4,1004,19,0.633333


In [18]:
item_properties["itemid"] = (
    item_properties["itemid"]
    .astype(str)
)

item_properties["property"] = (
    item_properties["property"]
    .astype(str)
)

item_properties["value"] = (
    item_properties["value"]
    .astype(str)
)

In [19]:
property_text = (

    item_properties

    .groupby("itemid")

    .apply(
        lambda x:
        " ".join(
            (
                x["property"]
                + " "
                + x["value"]
            ).tolist()
        )
    )

    .reset_index(
        name="product_text"
    )
)

display(
    property_text.head()
)

,itemid,product_text
0,1000,categoryid 18 available 1 790 n1774.245 6 n292...
1,1001,categoryid 14 available 1 790 n1559.428 6 n725...
2,1002,categoryid 7 available 0 790 n1114.505 6 n429....
3,1003,categoryid 25 available 0 790 n1397.447 6 n331...
4,1004,categoryid 23 available 1 790 n2284.286 6 n544...


In [20]:
product_catalog = (

    product_behavior

    .merge(
        property_text,
        on="itemid",
        how="left"
    )

)

product_catalog["product_text"] = (

    product_catalog["product_text"]
    .fillna("")

)

print(
    "Product catalog:",
    product_catalog.shape
)

display(
    product_catalog.head()
)

Product catalog: (600, 9)


,itemid,views,add_to_cart,transactions,conversion_rate,cart_rate,smoothed_conversion,popularity,product_text
0,1000,19,4,1,0.052632,0.210526,0.136364,0.633333,categoryid 18 available 1 790 n1774.245 6 n292...
1,1001,12,5,3,0.250000,0.416667,0.216216,0.400000,categoryid 14 available 1 790 n1559.428 6 n725...
2,1002,24,2,1,0.041667,0.083333,0.122449,0.800000,categoryid 7 available 0 790 n1114.505 6 n429....
3,1003,18,7,5,0.277778,0.388889,0.232558,0.600000,categoryid 25 available 0 790 n1397.447 6 n331...
4,1004,19,5,1,0.052632,0.263158,0.136364,0.633333,categoryid 23 available 1 790 n2284.286 6 n544...


In [21]:
property_counts = (
    item_properties["property"]
    .value_counts()
)

print(
    property_counts.head(50)
)

property
categoryid    1200
available     1200
790           1200
6             1200
364           1200
888           1200
283           1200
400           1200
776           1200
917           1200
Name: count, dtype: int64


In [22]:
keywords = [
    "category",
    "price",
    "brand",
    "color",
    "type",
    "name",
    "title"
]

for keyword in keywords:

    matches = [
        p
        for p in property_counts.index
        if keyword in p.lower()
    ]

    print(
        f"\n{keyword.upper()}:",
        matches
    )


CATEGORY: ['categoryid']

PRICE: []

BRAND: []

COLOR: []

TYPE: []

NAME: []

TITLE: []


In [23]:
category_candidates = [

    p
    for p in property_counts.index

    if (
        "category" in p.lower()
        or "cat" == p.lower()
    )
]

print(
    "Category candidates:",
    category_candidates
)

Category candidates: ['categoryid']


In [24]:
if len(category_candidates) > 0:

    selected_category_property = (
        category_candidates[0]
    )

    category_data = (

        item_properties[

            item_properties["property"]
            == selected_category_property

        ]

        [["itemid", "value"]]

        .drop_duplicates(
            "itemid"
        )

        .rename(
            columns={
                "value": "category"
            }
        )
    )

    product_catalog = (
        product_catalog
        .merge(
            category_data,
            on="itemid",
            how="left"
        )
    )

else:

    selected_category_property = None

    product_catalog["category"] = (
        "unknown"
    )

print(
    "Selected category property:",
    selected_category_property
)

Selected category property: categoryid


In [25]:
print(
    category_tree.columns.tolist()
)

display(
    category_tree.head(20)
)

['categoryid', 'parentid']


,categoryid,parentid
0,1,NaN
1,2,NaN
2,3,NaN
3,4,NaN
4,5,NaN
5,6,1.0
6,7,1.0
7,8,6.0
8,9,5.0
9,10,4.0


In [26]:
if (
    "categoryid" in category_tree.columns
    and
    "parentid" in category_tree.columns
):

    category_tree["categoryid"] = (
        category_tree["categoryid"]
        .astype(str)
    )

    category_tree["parentid"] = (
        category_tree["parentid"]
        .fillna("-1")
        .astype(str)
    )

    category_parent = dict(
        zip(
            category_tree["categoryid"],
            category_tree["parentid"]
        )
    )

else:

    category_parent = {}

In [27]:
def minmax_normalize(series):

    series = (
        pd.to_numeric(
            series,
            errors="coerce"
        )
        .fillna(0)
    )

    minimum = series.min()
    maximum = series.max()

    if maximum == minimum:

        return pd.Series(
            np.ones(len(series)),
            index=series.index
        )

    return (
        (series - minimum)
        /
        (maximum - minimum)
    )

In [28]:
product_catalog["quality_score"] = (

    minmax_normalize(
        product_catalog[
            "smoothed_conversion"
        ]
    )

)

product_catalog["popularity_score"] = (

    minmax_normalize(
        product_catalog[
            "views"
        ]
    )

)

product_catalog["discovery_score"] = (

    1
    -
    product_catalog[
        "popularity_score"
    ]

)

In [29]:
@dataclass
class Condition:

    category: str

    budget: float

    discovery_level: str = "medium"

    keywords: List[str] = field(
        default_factory=list
    )

    exclude_categories: List[str] = field(
        default_factory=list
    )

    strict_budget: bool = False

In [30]:
condition = Condition(

    category="unknown",

    budget=2000,

    discovery_level="high",

    keywords=[
        "fashion",
        "casual",
        "stylish",
        "unique"
    ],

    strict_budget=False
)

print(condition)

Condition(category='unknown', budget=2000, discovery_level='high', keywords=['fashion', 'casual', 'stylish', 'unique'], exclude_categories=[], strict_budget=False)


In [31]:
def filter_products(
    products,
    condition
):

    df = products.copy()

    # Must have interactions
    df = df[
        df["views"] > 0
    ]

    # Category filtering
    if (
        condition.category != "unknown"
        and "category" in df.columns
    ):

        df = df[
            df["category"]
            .fillna("")
            .str.lower()
            ==
            condition.category.lower()
        ]

    # Exclusions
    if (
        condition.exclude_categories
        and
        "category" in df.columns
    ):

        df = df[
            ~df["category"]
            .isin(
                condition.exclude_categories
            )
        ]

    return (
        df
        .reset_index(drop=True)
    )

In [32]:
def calculate_semantic_scores(
    products,
    condition
):

    query = " ".join(
        condition.keywords
    )

    if not query.strip():

        return np.zeros(
            len(products)
        )

    corpus = (
        products["product_text"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    if not any(
        text.strip()
        for text in corpus
    ):

        return np.zeros(
            len(products)
        )

    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english"
    )

    product_vectors = (
        vectorizer.fit_transform(
            corpus
        )
    )

    query_vector = (
        vectorizer.transform(
            [query]
        )
    )

    return cosine_similarity(
        query_vector,
        product_vectors
    )[0]

In [33]:
def category_match(
    row,
    condition
):

    if condition.category == "unknown":
        return 0.5

    if "category" not in row:
        return 0.5

    if pd.isna(
        row["category"]
    ):
        return 0.5

    return float(

        str(
            row["category"]
        ).lower()

        ==

        condition.category.lower()
    )

In [34]:
def budget_fit(
    price,
    budget,
    alpha=3
):

    if pd.isna(price):
        return 0.5

    if price <= budget:
        return 1.0

    excess = (
        price - budget
    ) / budget

    return np.exp(
        -alpha * excess
    )

In [35]:
def calculate_component_scores(
    products,
    condition
):

    df = products.copy()

    # CATEGORY
    df["category_score"] = df.apply(
        lambda row:
        category_match(
            row,
            condition
        ),
        axis=1
    )

    # SEMANTIC
    df["semantic_score"] = (
        calculate_semantic_scores(
            df,
            condition
        )
    )

    # QUALITY
    df["quality_score"] = (
        minmax_normalize(
            df["smoothed_conversion"]
        )
    )

    # DISCOVERY
    df["discovery_score"] = (

        1
        -
        minmax_normalize(
            df["views"]
        )
    )

    # BUDGET
    #
    # RetailRocket does not necessarily
    # expose a clean price column.
    #
    # Until we explicitly detect one,
    # use neutral budget score.

    if "price" in df.columns:

        df["budget_score"] = df.apply(

            lambda row:
            budget_fit(
                row["price"],
                condition.budget
            ),

            axis=1
        )

    else:

        df["budget_score"] = 0.5

    return df

In [36]:
WEIGHT_PROFILES = {

    "budget": {

        "category": 0.20,
        "budget": 0.45,
        "semantic": 0.20,
        "quality": 0.10,
        "discovery": 0.05

    },

    "balanced": {

        "category": 0.25,
        "budget": 0.25,
        "semantic": 0.25,
        "quality": 0.15,
        "discovery": 0.10

    },

    "high_discovery": {

        "category": 0.15,
        "budget": 0.15,
        "semantic": 0.30,
        "quality": 0.10,
        "discovery": 0.30

    }
}

In [37]:
def select_weights(condition):

    if (
        condition.discovery_level
        == "high"
    ):

        return WEIGHT_PROFILES[
            "high_discovery"
        ]

    elif (
        condition.discovery_level
        == "low"
    ):

        return WEIGHT_PROFILES[
            "budget"
        ]

    else:

        return WEIGHT_PROFILES[
            "balanced"
        ]

In [38]:
def calculate_final_score(
    df,
    weights
):

    df = df.copy()

    df["category_contribution"] = (

        df["category_score"]
        *
        weights["category"]
    )

    df["budget_contribution"] = (

        df["budget_score"]
        *
        weights["budget"]
    )

    df["semantic_contribution"] = (

        df["semantic_score"]
        *
        weights["semantic"]
    )

    df["quality_contribution"] = (

        df["quality_score"]
        *
        weights["quality"]
    )

    df["discovery_contribution"] = (

        df["discovery_score"]
        *
        weights["discovery"]
    )

    df["final_score"] = (

        df["category_contribution"]

        +

        df["budget_contribution"]

        +

        df["semantic_contribution"]

        +

        df["quality_contribution"]

        +

        df["discovery_contribution"]
    )

    return df

In [39]:
def rank_products(
    df,
    top_k=10
):

    ranked = (

        df
        .sort_values(
            "final_score",
            ascending=False
        )
        .reset_index(drop=True)
    )

    ranked["rank"] = (
        np.arange(
            len(ranked)
        ) + 1
    )

    return ranked.head(
        top_k
    )

In [40]:
def recommend_products(
    products,
    condition,
    top_k=10
):

    # 1. FILTER
    eligible = filter_products(
        products,
        condition
    )

    if len(eligible) == 0:

        print(
            "No eligible products."
        )

        return pd.DataFrame()

    # 2. COMPONENT SCORES
    scored = (
        calculate_component_scores(
            eligible,
            condition
        )
    )

    # 3. DYNAMIC WEIGHTS
    weights = select_weights(
        condition
    )

    # 4. FINAL SCORE
    scored = (
        calculate_final_score(
            scored,
            weights
        )
    )

    # 5. RANK
    ranked = rank_products(
        scored,
        top_k
    )

    return ranked

In [41]:
recommendations = recommend_products(

    product_catalog,

    condition,

    top_k=10
)

print(
    "Recommendations:",
    len(recommendations)
)

Recommendations: 10


In [42]:
display_columns = [

    "rank",
    "itemid",
    "final_score",
    "category_score",
    "budget_score",
    "semantic_score",
    "quality_score",
    "discovery_score"
]

display(
    recommendations[
        [
            c
            for c in display_columns
            if c in recommendations.columns
        ]
    ]
)

,rank,itemid,final_score,category_score,budget_score,semantic_score,quality_score,discovery_score
0,1,1114,0.532553,0.5,0.5,0.0,0.825535,1.000000
1,2,1434,0.515955,0.5,0.5,0.0,0.789985,0.956522
2,3,1457,0.509091,0.5,0.5,0.0,0.590909,1.000000
3,4,1151,0.504819,0.5,0.5,0.0,0.939496,0.869565
4,5,1032,0.488200,0.5,0.5,0.0,0.903743,0.826087
5,6,1045,0.483368,0.5,0.5,0.0,0.724981,0.869565
6,7,1314,0.477483,0.5,0.5,0.0,0.535703,0.913043
7,8,1082,0.472642,0.5,0.5,0.0,0.617723,0.869565
8,9,1212,0.472642,0.5,0.5,0.0,0.617723,0.869565
9,10,1347,0.472642,0.5,0.5,0.0,0.617723,0.869565


In [43]:
def explain_product(row):

    contributions = {

        "Category Match":
            row[
                "category_contribution"
            ],

        "Budget Fit":
            row[
                "budget_contribution"
            ],

        "Semantic / Style Fit":
            row[
                "semantic_contribution"
            ],

        "Quality":
            row[
                "quality_contribution"
            ],

        "Discovery":
            row[
                "discovery_contribution"
            ]
    }

    contributions = dict(
        sorted(
            contributions.items(),
            key=lambda x: x[1],
            reverse=True
        )
    )

    print(
        "\nProduct:",
        row["itemid"]
    )

    print(
        f"Final Score: "
        f"{row['final_score']:.4f}"
    )

    print(
        "\nContributions:"
    )

    for component, value in (
        contributions.items()
    ):

        print(
            f"{component:<25}"
            f"+{value:.4f}"
        )

    print(
        "\nVerification:"
    )

    print(
        "Contribution sum:",
        round(
            sum(
                contributions.values()
            ),
            4
        )
    )

    print(
        "Final score:",
        round(
            row["final_score"],
            4
        )
    )

In [44]:
for i in range(
    min(3, len(recommendations))
):

    explain_product(
        recommendations.iloc[i]
    )


Product: 1114
Final Score: 0.5326

Contributions:
Discovery                +0.3000
Quality                  +0.0826
Category Match           +0.0750
Budget Fit               +0.0750
Semantic / Style Fit     +0.0000

Verification:
Contribution sum: 0.5326
Final score: 0.5326

Product: 1434
Final Score: 0.5160

Contributions:
Discovery                +0.2870
Quality                  +0.0790
Category Match           +0.0750
Budget Fit               +0.0750
Semantic / Style Fit     +0.0000

Verification:
Contribution sum: 0.516
Final score: 0.516

Product: 1457
Final Score: 0.5091

Contributions:
Discovery                +0.3000
Category Match           +0.0750
Budget Fit               +0.0750
Quality                  +0.0591
Semantic / Style Fit     +0.0000

Verification:
Contribution sum: 0.5091
Final score: 0.5091


In [45]:
def generate_api_output(
    recommendations,
    condition
):

    weights = select_weights(
        condition
    )

    results = []

    for _, row in (
        recommendations.iterrows()
    ):

        result = {

            "rank":
                int(row["rank"]),

            "item_id":
                str(row["itemid"]),

            "product_intelligence_score":
                round(
                    float(
                        row[
                            "final_score"
                        ]
                    ),
                    4
                ),

            "components": {

                "category_match":
                    round(
                        float(
                            row[
                                "category_score"
                            ]
                        ),
                        4
                    ),

                "budget_fit":
                    round(
                        float(
                            row[
                                "budget_score"
                            ]
                        ),
                        4
                    ),

                "semantic_fit":
                    round(
                        float(
                            row[
                                "semantic_score"
                            ]
                        ),
                        4
                    ),

                "quality":
                    round(
                        float(
                            row[
                                "quality_score"
                            ]
                        ),
                        4
                    ),

                "discovery":
                    round(
                        float(
                            row[
                                "discovery_score"
                            ]
                        ),
                        4
                    )
            },

            "contributions": {

                "category_match":
                    round(
                        float(
                            row[
                                "category_contribution"
                            ]
                        ),
                        4
                    ),

                "budget_fit":
                    round(
                        float(
                            row[
                                "budget_contribution"
                            ]
                        ),
                        4
                    ),

                "semantic_fit":
                    round(
                        float(
                            row[
                                "semantic_contribution"
                            ]
                        ),
                        4
                    ),

                "quality":
                    round(
                        float(
                            row[
                                "quality_contribution"
                            ]
                        ),
                        4
                    ),

                "discovery":
                    round(
                        float(
                            row[
                                "discovery_contribution"
                            ]
                        ),
                        4
                    )
            },

            "weights": weights
        }

        results.append(result)

    return {

        "condition": {

            "category":
                condition.category,

            "budget":
                condition.budget,

            "discovery_level":
                condition.discovery_level,

            "keywords":
                condition.keywords
        },

        "recommendations":
            results
    }

In [46]:
api_output = generate_api_output(
    recommendations,
    condition
)

api_output

{'condition': {'category': 'unknown',
  'budget': 2000,
  'discovery_level': 'high',
  'keywords': ['fashion', 'casual', 'stylish', 'unique']},
 'recommendations': [{'rank': 1,
   'item_id': '1114',
   'product_intelligence_score': 0.5326,
   'components': {'category_match': 0.5,
    'budget_fit': 0.5,
    'semantic_fit': 0.0,
    'quality': 0.8255,
    'discovery': 1.0},
   'contributions': {'category_match': 0.075,
    'budget_fit': 0.075,
    'semantic_fit': 0.0,
    'quality': 0.0826,
    'discovery': 0.3},
   'weights': {'category': 0.15,
    'budget': 0.15,
    'semantic': 0.3,
    'quality': 0.1,
    'discovery': 0.3}},
  {'rank': 2,
   'item_id': '1434',
   'product_intelligence_score': 0.516,
   'components': {'category_match': 0.5,
    'budget_fit': 0.5,
    'semantic_fit': 0.0,
    'quality': 0.79,
    'discovery': 0.9565},
   'contributions': {'category_match': 0.075,
    'budget_fit': 0.075,
    'semantic_fit': 0.0,
    'quality': 0.079,
    'discovery': 0.287},
   'weight

In [47]:
product_catalog.to_csv(
    "/content/retailrocket_enriched_catalog.csv",
    index=False
)

recommendations.to_csv(
    "/content/product_intelligence_results.csv",
    index=False
)

print(
    "Files saved successfully."
)

Files saved successfully.
